# Seam fidelity vs autoregressive step

Does a stitched field carry a signature at the seam that it does not carry elsewhere,
and does that signature grow as the rollout advances?

Three measures, each computed per autoregressive step:

1. **RMSE vs truth** — the headline error, but it mixes the seam in with ordinary
   forecast error, so on its own it is the weakest of the three.
2. **Derivative jump** — a hard stitch is a kink, and a kink is a delta function in
   the derivative. This is the most direct seam signature.
3. **High-k power ratio** — averaging decorrelated small scales destroys them, so an
   over-broad blend shows up as a *deficit* (ratio < 1) localised at the seam.

Every measure is computed at the real seam **and** at matched pseudo-seams drawn
through undisturbed interior, and the **anomaly** between them is the result. Ocean
energy varies across the domain, so a raw curve that rises near the seam may just be
sampling a more energetic region; only the anomaly controls for that.

### On pseudo-seam placement

The strip-length worry resolves itself if you stop thinking in strips. Fix an
analysis half-width `W` and place the pseudo-seam centres so that `±W` about each of
them clears both the real seam and the padded exterior. `seam_windows` does this and
refuses widths that would violate it, so a contaminated comparison is not
expressible. With the live geometry — 720 cells, seam at 360, 16-cell overlap —
`W = 64` puts pseudo-seams at 180 and 540, sampling [116, 244] and [476, 604].
Neither reaches the seam at [352, 368] nor the constant-padded edge.

The one extra care needed is the *other* seam: the vertical-seam curve must not sample
cells near the horizontal seam. `orthogonal_exclusion` drops that band, identically
for the real and pseudo windows, which is what keeps them matched.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from ocean_emulators.tile_diagnostics import (
    derivative_jump,
    high_k_power_ratio,
    seam_anomaly,
    seam_windows,
)

plt.rcParams['figure.dpi'] = 110

## Load

In [ ]:
# ============== LOAD LLC TRUTH ==============
# The canonical 2x2 union is face 1, i=[2880:3600), j=[720:1440) -- the same
# 720x720 Agulhas patch the single-patch model was trained on.
llc_full = xr.open_zarr('/orcd/data/abodner/003/LLC4320/LLC4320', consolidated=False).isel(
    face=1,
    i=slice(2880, 3600),
    i_g=slice(2880, 3600),
    j=slice(720, 1440),
    j_g=slice(720, 1440),
)

# ============== LOAD TILED ROLLOUTS ==============
# Rung 2 of the ladder is the hard-crop control (BLEND=false); rung 3 is the blend.
# Both write a canonical 720x720 predictions.zarr from tiled_eval.
run_configs = [
    {
        'name': 'hard crop (no blend)',
        'key': 'crop',
        'path': '/orcd/data/abodner/002/cody/inference_patch/'
                '2026-08-09-eval:Samudra_LLC:4-tile-hardcrop/predictions.zarr',
        'desc': 'rung 2 -- tiles stepped independently, cut and stitched',
    },
    {
        'name': 'quintic blend',
        'key': 'quintic',
        'path': '/orcd/data/abodner/002/cody/inference_patch/'
                '2026-08-09-eval:Samudra_LLC:4-tile-blended-rollout/predictions.zarr',
        'desc': 'rung 3 -- smootherstep partition of unity at inference only',
    },
    # {
    #     'name': 'kbd blend (STRATA-like)',
    #     'key': 'kbd',
    #     'path': '.../predictions.zarr',
    #     'desc': 'rung 3 with the Kaiser-Bessel-derived window',
    # },
]

runs_raw = {}
for cfg in run_configs:
    runs_raw[cfg['key']] = xr.open_zarr(cfg['path'], consolidated=True)
    print(f"Loaded {cfg['name']}: {cfg['desc']}")
    print(f"   {dict(runs_raw[cfg['key']].sizes)}")

In [ ]:
# ============== TIME MATCHING ==============
def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(
            int(t.year), int(t.month), int(t.day),
            int(t.hour), int(t.minute), int(t.second)
        )
        if hasattr(t, 'year')
        else pd.Timestamp(t).floor('s')
        for t in times
    ])

llc_times_norm = normalize_times(llc_full.time.values)

common_times = llc_times_norm
for cfg in run_configs:
    common_times = common_times.intersection(
        normalize_times(runs_raw[cfg['key']].time.values)
    )
common_times = common_times.sort_values()

llc_patch = llc_full.isel(time=llc_times_norm.isin(common_times))
runs = {}
for cfg in run_configs:
    run_times = normalize_times(runs_raw[cfg['key']].time.values)
    runs[cfg['key']] = runs_raw[cfg['key']].isel(time=run_times.isin(common_times))

n_steps = len(common_times)
print(f'{n_steps} common autoregressive steps: {common_times[0]} -> {common_times[-1]}')

## Seam geometry

The tiles overlap by 16 cells, so both seams sit at canonical index 360.
Everything below is computed against these windows.

In [ ]:
# ============== SEAM GEOMETRY KNOBS ==============
CANONICAL = 720          # canonical grid extent on each axis
SEAM_CENTRE = 360        # 16-cell overlap centred here on both axes
HALF_WIDTH = 64          # analysis half-width; 4x the overlap, clears everything

# The vertical seam runs along i; the horizontal seam along j. Each excludes a
# band around the other so the two do not contaminate each other's curves.
windows_i = seam_windows(
    axis='i', seam_centre=SEAM_CENTRE, extent=CANONICAL,
    half_width=HALF_WIDTH, orthogonal_seam_centre=SEAM_CENTRE,
)
windows_j = seam_windows(
    axis='j', seam_centre=SEAM_CENTRE, extent=CANONICAL,
    half_width=HALF_WIDTH, orthogonal_seam_centre=SEAM_CENTRE,
)

for label, w in [('vertical (i)', windows_i), ('horizontal (j)', windows_j)]:
    print(f'{label:16s} real seam at {w.real_centre}, '
          f'pseudo-seams at {w.pseudo_centres}, +/-{w.half_width} cells')
    print(f'{"":16s} orthogonal exclusion: {w.orthogonal_exclusion}')

## Variable selection

In [ ]:
# ============== DIAGNOSTIC KNOBS ==============
# Surface fields show submesoscale structure most clearly, which is where the
# blending question actually bites.
DIAG_VARS = ['Theta_0', 'Salt_0', 'U_0', 'V_0']
HIGH_K_FRACTION = 0.5      # top half of the wavenumber band counts as "high k"
CURVE_STEPS = None         # steps to draw curves for; None -> first, middle, last

# Band used for the headline numbers. Wider than the 8-cell half-overlap on
# purpose: derivative_jump reports a forward difference padded on the right, so a
# hard stitch's delta lands about a cell to the low side of the overlap edge and a
# knife-edge band of +/-8 would miss it entirely.
SEAM_BAND = 12

if CURVE_STEPS is None:
    CURVE_STEPS = sorted({0, n_steps // 2, n_steps - 1})
print('drawing curves at steps', CURVE_STEPS)


def llc_surface(var):
    """Pull the matching LLC truth field for a flat prediction channel name."""
    base, _, level = var.rpartition('_')
    return llc_patch[base].isel(k=int(level)).values


def run_surface(key, var):
    return runs[key][var].values

## Per-step anomalies

For each run, each variable and each step we build the three fields, reduce them to
curves against signed distance from the seam, and subtract the matched pseudo-seam
curve.

In [ ]:
# ============== COMPUTE SEAM ANOMALIES PER STEP ==============
# results[key][var][measure] -> array of shape (n_steps, 2*HALF_WIDTH+1)
MEASURES = ['rmse', 'djump', 'highk']
results = {cfg['key']: {var: {m: [] for m in MEASURES} for var in DIAG_VARS}
           for cfg in run_configs}
raw_curves = {cfg['key']: {var: {m: [] for m in MEASURES} for var in DIAG_VARS}
              for cfg in run_configs}

for var in DIAG_VARS:
    truth_all = llc_surface(var)
    print(f'Computing seam anomalies for {var}...')
    for cfg in run_configs:
        key = cfg['key']
        pred_all = run_surface(key, var)
        for step in range(n_steps):
            truth = truth_all[step]
            pred = pred_all[step]

            error = np.abs(pred - truth)
            jump = np.abs(derivative_jump(pred, axis='i')) - np.abs(
                derivative_jump(truth, axis='i')
            )
            ratio = np.broadcast_to(
                high_k_power_ratio(pred, truth, axis='i',
                                   high_k_fraction=HIGH_K_FRACTION),
                pred.shape,
            )

            for measure, field, reduce in [
                ('rmse', error, 'rms'),
                ('djump', jump, 'absmean'),
                ('highk', ratio, 'mean'),
            ]:
                out = seam_anomaly(field, windows_i, reduce=reduce)
                results[key][var][measure].append(out['anomaly'])
                raw_curves[key][var][measure].append(
                    np.stack([out['seam'], out['pseudo']])
                )

        for measure in MEASURES:
            results[key][var][measure] = np.stack(results[key][var][measure])
            raw_curves[key][var][measure] = np.stack(raw_curves[key][var][measure])

offsets = np.arange(-HALF_WIDTH, HALF_WIDTH + 1)
print('done:', results[run_configs[0]['key']][DIAG_VARS[0]]['rmse'].shape,
      '(step, offset)')

## Curves vs signed distance from the seam

Solid is the real seam, dashed the matched pseudo-seam. Where they lie on top of each
other there is no seam signature; the gap between them is the whole result.

In [ ]:
# ============== CURVES: REAL SEAM vs PSEUDO-SEAM ==============
MEASURE_LABELS = {
    'rmse': 'RMSE vs truth',
    'djump': '|d/di| excess over truth',
    'highk': f'high-k power ratio (top {HIGH_K_FRACTION:.0%})',
}

for var in DIAG_VARS:
    fig, axes = plt.subplots(
        len(MEASURES), len(CURVE_STEPS),
        figsize=(4.0 * len(CURVE_STEPS), 3.0 * len(MEASURES)),
        squeeze=False, sharex=True,
    )
    for row, measure in enumerate(MEASURES):
        for col, step in enumerate(CURVE_STEPS):
            ax = axes[row][col]
            for cfg in run_configs:
                seam, pseudo = raw_curves[cfg['key']][var][measure][step]
                line, = ax.plot(offsets, seam, lw=1.6, label=f"{cfg['name']} (seam)")
                ax.plot(offsets, pseudo, lw=1.1, ls='--', color=line.get_color(),
                        alpha=0.7, label=f"{cfg['name']} (pseudo)")
            ax.axvspan(-SEAM_BAND, SEAM_BAND, color='0.85', zorder=0)
            ax.axvline(0, color='k', lw=0.6)
            if measure == 'highk':
                ax.axhline(1.0, color='k', lw=0.5, ls=':')
            if row == 0:
                ax.set_title(f'step {step}')
            if col == 0:
                ax.set_ylabel(MEASURE_LABELS[measure])
            if row == len(MEASURES) - 1:
                ax.set_xlabel('signed distance from seam (cells)')
    axes[0][0].legend(fontsize=7, framealpha=0.9)
    fig.suptitle(f'{var}: seam vs matched pseudo-seam (shaded = 16-cell overlap)')
    fig.tight_layout()
    plt.show()

## Time series: does the seam signature grow with the rollout?

The value at offset 0 is the sharpest single number. A flat line means blending is
holding; a rising line means the seam is compounding autoregressively, which is the
failure mode that matters most.

In [ ]:
# ============== AT-SEAM ANOMALY vs AUTOREGRESSIVE STEP ==============
# A kink is a delta function, so the derivative jump must be summarised with a
# MAX over the band. Averaging a single spike divides it by the window width,
# which makes a hard stitch score better than a smooth blend that spreads the
# same total variation over many cells -- backwards. RMSE and the high-k ratio
# are diffuse, so a mean is right for them.
overlap_band = np.abs(offsets) <= SEAM_BAND
BAND_REDUCERS = {
    'rmse': lambda a: a[:, overlap_band].mean(axis=1),
    'djump': lambda a: a[:, overlap_band].max(axis=1),
    'highk': lambda a: a[:, overlap_band].mean(axis=1),
}

fig, axes = plt.subplots(
    len(MEASURES), len(DIAG_VARS),
    figsize=(3.6 * len(DIAG_VARS), 2.8 * len(MEASURES)),
    squeeze=False, sharex=True,
)
for row, measure in enumerate(MEASURES):
    for col, var in enumerate(DIAG_VARS):
        ax = axes[row][col]
        for cfg in run_configs:
            anomaly = results[cfg['key']][var][measure]
            ax.plot(BAND_REDUCERS[measure](anomaly), lw=1.6, label=cfg['name'])
        ax.axhline(0.0, color='k', lw=0.5, ls=':')
        if row == 0:
            ax.set_title(var)
        if col == 0:
            ax.set_ylabel(MEASURE_LABELS[measure] + '\nanomaly')
        if row == len(MEASURES) - 1:
            ax.set_xlabel('autoregressive step')
axes[0][0].legend(fontsize=7)
fig.suptitle(f'Seam-minus-pseudo-seam anomaly vs rollout step '
             f'(max over +/-{SEAM_BAND} cells for the jump, mean for the rest)')
fig.tight_layout()
plt.show()

## Time-independent view

Averaging over the rollout is worth doing on its own: the anomaly is a small
difference of two noisy curves, and the time mean is where a weak but real signature
becomes visible. Step 0 is shown separately because a single-step signature is a
property of the operator, whereas a growing one is a property of the feedback.

In [ ]:
# ============== TIME-MEAN CURVES, AND STEP 0 ALONE ==============
fig, axes = plt.subplots(
    len(MEASURES), len(DIAG_VARS),
    figsize=(3.6 * len(DIAG_VARS), 2.8 * len(MEASURES)),
    squeeze=False, sharex=True,
)
for row, measure in enumerate(MEASURES):
    for col, var in enumerate(DIAG_VARS):
        ax = axes[row][col]
        for cfg in run_configs:
            anomaly = results[cfg['key']][var][measure]
            line, = ax.plot(offsets, anomaly.mean(axis=0), lw=1.8,
                            label=f"{cfg['name']} (time mean)")
            ax.plot(offsets, anomaly[0], lw=1.0, ls='--', alpha=0.65,
                    color=line.get_color(), label=f"{cfg['name']} (step 0)")
        ax.axvspan(-SEAM_BAND, SEAM_BAND, color='0.85', zorder=0)
        ax.axhline(0, color='k', lw=0.5, ls=':')
        if row == 0:
            ax.set_title(var)
        if col == 0:
            ax.set_ylabel(MEASURE_LABELS[measure] + '\nanomaly')
        if row == len(MEASURES) - 1:
            ax.set_xlabel('signed distance from seam (cells)')
axes[0][0].legend(fontsize=7)
fig.suptitle('Time-mean seam anomaly, with step 0 for comparison')
fig.tight_layout()
plt.show()

## Maps: look at the seam directly

Curves can hide a localised artefact. The derivative field makes a kink visible by
eye, which is the cheapest sanity check that the curves above are measuring what we
think they are.

In [ ]:
# ============== DERIVATIVE MAPS AROUND THE SEAM ==============
MAP_VAR = DIAG_VARS[0]
MAP_STEP = CURVE_STEPS[-1]
MAP_HALF = 48   # zoom half-width around the seam crossing

sl = slice(SEAM_CENTRE - MAP_HALF, SEAM_CENTRE + MAP_HALF)
panels = [('LLC truth', np.abs(derivative_jump(llc_surface(MAP_VAR)[MAP_STEP], axis='i')))]
for cfg in run_configs:
    panels.append((
        cfg['name'],
        np.abs(derivative_jump(run_surface(cfg['key'], MAP_VAR)[MAP_STEP], axis='i')),
    ))

vmax = np.nanpercentile(np.stack([p[1][sl, sl] for p in panels]), 99)
fig, axes = plt.subplots(1, len(panels), figsize=(4.0 * len(panels), 4.0), squeeze=False)
for ax, (label, field) in zip(axes[0], panels):
    im = ax.imshow(field[sl, sl], origin='lower', vmin=0, vmax=vmax, cmap='magma')
    ax.axvline(MAP_HALF, color='cyan', lw=0.7, ls='--')
    ax.axhline(MAP_HALF, color='cyan', lw=0.7, ls='--')
    ax.set_title(label, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes[0], shrink=0.8, label=f'|d{MAP_VAR}/di|')
fig.suptitle(f'{MAP_VAR} derivative magnitude at step {MAP_STEP} '
             f'(cyan = seam centre lines)')
plt.show()

## Summary table

One number per run per measure: the time-mean anomaly averaged over the overlap band.
Closer to zero is better for RMSE and the derivative jump; closer to zero is also
better for the high-k ratio anomaly, where a **negative** value means the stitch has
eaten variance the truth still has.

In [ ]:
# ============== SUMMARY ==============
rows = []
for cfg in run_configs:
    for var in DIAG_VARS:
        row = {'run': cfg['name'], 'var': var}
        for measure in MEASURES:
            anomaly = results[cfg['key']][var][measure]
            reduced = BAND_REDUCERS[measure](anomaly)
            row[f'{measure}_mean'] = float(reduced.mean())
            row[f'{measure}_final'] = float(reduced[-1])
        rows.append(row)

summary = pd.DataFrame(rows)
pd.set_option('display.float_format', lambda v: f'{v: .4e}')
display(summary)